In [2]:
from google.colab import files
uploaded = files.upload()

Saving Reviews.csv to Reviews.csv


In [3]:
import pandas as pd

df = pd.read_csv("Reviews.csv", nrows=10000)
df = df[['Score', 'Text']].dropna()

print(df.shape)
print(df['Score'].value_counts())
df.head()

(10000, 2)
Score
5    6183
4    1433
1     932
3     862
2     590
Name: count, dtype: int64


,Score,Text
0,5,I have bought several of the Vitality canned d...
1,1,Product arrived labeled as Jumbo Salted Peanut...
2,4,This is a confection that has been around a fe...
3,2,If you are looking for the secret ingredient i...
4,5,Great taffy at a great price. There was a wid...


In [4]:
df['review_length'] = df['Text'].apply(lambda x: len(x.split()))

print(df.groupby('Score')['review_length'].mean().round(1))

Score
1    85.4
2    88.8
3    90.1
4    86.2
5    69.4
Name: review_length, dtype: float64


In [5]:
def label_sentiment(score):
    if score >= 4:
        return 'Positive'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Negative'

df['sentiment'] = df['Score'].apply(label_sentiment)
print(df['sentiment'].value_counts())

sentiment
Positive    7616
Negative    1522
Neutral      862
Name: count, dtype: int64


In [6]:
import scipy.stats as stats

correlation, pvalue = stats.pearsonr(df['review_length'], df['Score'])
print(f"Correlation: {correlation:.4f}")
print(f"P-value: {pvalue:.4f}")

if abs(correlation) < 0.3:
    print("Finding: WEAK correlation — longer reviews do NOT strongly predict sentiment ❌")
else:
    print("Finding: STRONG correlation found ✅")

Correlation: -0.0975
P-value: 0.0000
Finding: WEAK correlation — longer reviews do NOT strongly predict sentiment ❌


In [7]:
import plotly.express as px
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import base64
from io import BytesIO

# helper
def wc_to_base64(text, colormap="viridis"):
    from wordcloud import WordCloud
    wc = WordCloud(width=700, height=300,
                   background_color="white",
                   colormap=colormap).generate(text)
    buf = BytesIO()
    wc.to_image().save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

COLOR = {'Positive':'#2ecc71','Neutral':'#95a5a6','Negative':'#e74c3c'}

# Chart 1 - sentiment distribution
fig1 = px.pie(df, names='sentiment', title='Sentiment Distribution',
              color='sentiment', color_discrete_map=COLOR, hole=0.4)
fig1.update_layout(paper_bgcolor='rgba(0,0,0,0)')

# Chart 2 - avg review length per sentiment
avg_len = df.groupby('sentiment')['review_length'].mean().reset_index()
fig2 = px.bar(avg_len, x='sentiment', y='review_length',
              color='sentiment', color_discrete_map=COLOR,
              title='Avg Review Length per Sentiment',
              text='review_length')
fig2.update_traces(texttemplate='%{text:.0f} words', textposition='outside')
fig2.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   plot_bgcolor='rgba(0,0,0,0)', showlegend=False)

# Chart 3 - scatter plot
fig3 = px.scatter(df.sample(2000), x='review_length', y='Score',
                  color='sentiment', color_discrete_map=COLOR,
                  title='Review Length vs Star Rating (sample of 2000)',
                  opacity=0.5)
fig3.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   plot_bgcolor='rgba(0,0,0,0)')

# Chart 4 - score distribution
fig4 = px.histogram(df, x='Score', title='Score Distribution',
                    color_discrete_sequence=['#2e86c1'])
fig4.update_layout(paper_bgcolor='rgba(0,0,0,0)',
                   plot_bgcolor='rgba(0,0,0,0)')

# word clouds
wc_pos = wc_to_base64(" ".join(df[df['sentiment']=='Positive']['Text'].dropna()), "Greens")
wc_neg = wc_to_base64(" ".join(df[df['sentiment']=='Negative']['Text'].dropna()), "Reds")

# correlation result
corr_color = "#e74c3c" if abs(correlation) < 0.3 else "#2ecc71"
corr_label = "Weak — No significant pattern found" if abs(correlation) < 0.3 else "Strong correlation found"

pie_html  = fig1.to_html(full_html=False, include_plotlyjs=False)
bar_html  = fig2.to_html(full_html=False, include_plotlyjs=False)
scat_html = fig3.to_html(full_html=False, include_plotlyjs=False)
hist_html = fig4.to_html(full_html=False, include_plotlyjs=False)

total = len(df)
pos = len(df[df['sentiment']=='Positive'])
neg = len(df[df['sentiment']=='Negative'])
neu = len(df[df['sentiment']=='Neutral'])

html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>Amazon Reviews — Failed Analysis Dashboard</title>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ font-family:'Segoe UI',sans-serif; background:#f0f2f5; color:#333; }}
  header {{
    background: linear-gradient(135deg,#1a5276,#2e86c1);
    color:white; padding:30px 40px;
  }}
  header h1 {{ font-size:1.8rem; font-weight:700; }}
  header p  {{ font-size:0.95rem; opacity:0.85; margin-top:6px; }}
  .finding {{
    margin:24px 40px 0;
    background:white; border-radius:12px;
    padding:20px 28px;
    border-left:6px solid {corr_color};
    box-shadow:0 2px 8px rgba(0,0,0,.07);
  }}
  .finding h2 {{ font-size:1rem; color:#555; margin-bottom:6px; }}
  .finding .result {{ font-size:1.4rem; font-weight:700; color:{corr_color}; }}
  .finding .sub {{ font-size:0.85rem; color:#888; margin-top:4px; }}
  .kpi-row {{
    display:grid; grid-template-columns:repeat(4,1fr);
    gap:16px; padding:24px 40px 0;
  }}
  .kpi {{
    background:white; border-radius:12px;
    padding:20px 24px; text-align:center;
    box-shadow:0 2px 8px rgba(0,0,0,.07);
  }}
  .kpi .num {{ font-size:2rem; font-weight:700; }}
  .kpi .lbl {{ font-size:0.85rem; color:#777; margin-top:4px; }}
  .kpi.total .num {{ color:#2e86c1; }}
  .kpi.pos   .num {{ color:#2ecc71; }}
  .kpi.neu   .num {{ color:#95a5a6; }}
  .kpi.neg   .num {{ color:#e74c3c; }}
  .charts-row {{
    display:grid; grid-template-columns:1fr 1fr;
    gap:16px; padding:24px 40px 0;
  }}
  .card {{
    background:white; border-radius:12px; padding:20px;
    box-shadow:0 2px 8px rgba(0,0,0,.07);
  }}
  .card h3 {{ font-size:1rem; color:#555; margin-bottom:12px; }}
  .card img {{ width:100%; border-radius:8px; }}
  .wc-grid {{
    display:grid; grid-template-columns:1fr 1fr;
    gap:16px; padding:24px 40px;
  }}
  footer {{
    text-align:center; padding:20px;
    font-size:0.8rem; color:#aaa;
  }}
</style>
</head>
<body>

<header>
  <h1>🔍 Amazon Food Reviews — "Failed" Analysis</h1>
  <p>Hypothesis: Do longer reviews indicate stronger sentiment? · 10,000 reviews analysed</p>
</header>

<div class="finding">
  <h2>📊 Hypothesis Result</h2>
  <div class="result">{corr_label}</div>
  <div class="sub">Pearson Correlation: {correlation:.4f} · P-value: {pvalue:.4f} · This is a valid and honest analytical finding.</div>
</div>

<div class="kpi-row">
  <div class="kpi total"><div class="num">{total:,}</div><div class="lbl">Total Reviews</div></div>
  <div class="kpi pos">  <div class="num">{pos:,}</div>  <div class="lbl">Positive</div></div>
  <div class="kpi neu">  <div class="num">{neu:,}</div>  <div class="lbl">Neutral</div></div>
  <div class="kpi neg">  <div class="num">{neg:,}</div>  <div class="lbl">Negative</div></div>
</div>

<div class="charts-row">
  <div class="card">{pie_html}</div>
  <div class="card">{bar_html}</div>
</div>

<div class="charts-row">
  <div class="card">{scat_html}</div>
  <div class="card">{hist_html}</div>
</div>

<div class="wc-grid">
  <div class="card"><h3>💚 Positive Reviews — Common Words</h3>
    <img src="data:image/png;base64,{wc_pos}"/></div>
  <div class="card"><h3>❤️ Negative Reviews — Common Words</h3>
    <img src="data:image/png;base64,{wc_neg}"/></div>
</div>

<footer>Built by Varshini · Amazon Food Reviews Analysis · 2026</footer>
</body>
</html>
"""

with open("amazon_failed_analysis.html", "w", encoding="utf-8") as f:
    f.write(html)

from google.colab import files
files.download("amazon_failed_analysis.html")

print("Dashboard downloaded ✅ Open in browser!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dashboard downloaded ✅ Open in browser!
